# 05v2 - A2: slot-attention model, v1

**Plan item A2** (see [[project-rsna-phase-status]] /
docs/superpowers/specs/2026-08-26-a2-slot-attention-model-design.md).
Self-contained per this project's Kaggle constraint (no `import src` -
confirmed 2026-08-26) - every function below is a hand-kept copy of the
matching `src/` function where one exists (`src/data.py::load_published_labels`,
`src/data.py::load_training_labels`, `src/model.py::masked_finding_attention`)
- keep them in sync manually if either side changes.

**v1 scope:** DINOv2-small + per-finding masked attention over the 6 A3
slots, centre anchor only (`group_index=1`), fold 0 only, no Tier B
items (no `pos_weight`, no `is_gold` upweighting, no EMA, no
augmentation, no compartment-aware attention).

In [ ]:
import hashlib
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
ON_KAGGLE = _KAGGLE_RAW.exists()
if not ON_KAGGLE:
    raise RuntimeError(
        "This notebook needs the full DICOM tree + GPU + the A3 cache "
        "attached - run on Kaggle, not locally."
    )
RAW_DIR = _KAGGLE_RAW
CACHE_DIR = Path("/kaggle/input/datasets/alherma7/cache-stevenleehans-rsna/cache")

# llm_labels_v4_blend.csv is NOT part of the official competition mount
# (RAW_DIR is read-only, Kaggle-provided data only) - it's a separate
# file the user uploaded as their own small Kaggle Dataset and attached
# to this kernel. EDIT this path to match wherever it actually lands
# under /kaggle/input/ once attached (same convention as CACHE_DIR
# above - check with `!ls /kaggle/input` if unsure).
PUBLISHED_LABELS_PATH = Path("/kaggle/input/llm-labels-v4-blend/llm_labels_v4_blend.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("CACHE_DIR:", CACHE_DIR, "exists:", CACHE_DIR.exists())
print("PUBLISHED_LABELS_PATH:", PUBLISHED_LABELS_PATH, "exists:", PUBLISHED_LABELS_PATH.exists())

FINDINGS = [
    "acl_injury", "mcl_injury", "medial_meniscus_tear", "lateral_meniscus_tear",
    "oa_medial_compartment", "oa_lateral_compartment", "oa_patellofemoral_compartment",
    "effusion", "synovitis", "bakers_cyst", "bone_contusion", "fracture",
]
OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL", "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus", "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA", "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA", "effusion": "Effusion",
    "synovitis": "Synovitis", "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion", "fracture": "Fracture",
}
SLOT_NAMES = ["SAG_FLUID_FS", "COR_FLUID_FS", "AX_FLUID_FS", "SAG_FLUID_NOFS", "COR_T1", "SAG_T1"]
SLOT_CACHE_GROUP_SIZE = 3
GROUP_INDEX = 1  # centre anchor, per the approved A2 spec section 1
CV_FOLDS = 4
TRAIN_SHARDS = [f"train.s{i:02d}of04" for i in range(4)]

## Labels: gold official values + A1a' published set for weak studies

Note: unlike `src/data.py::load_published_labels()` (which reads
`raw_dir/_published_labels/...` - a local-only convention, since the
user extracted this file directly under `data/raw/` locally), this
notebook reads `PUBLISHED_LABELS_PATH` directly - `RAW_DIR` on Kaggle is
the read-only official competition mount and does NOT include this
file; it's a separate small Dataset the user uploaded and attached
themselves.

In [ ]:
def load_published_labels(path):
    published = pd.read_csv(path)
    label_cols = list(OFFICIAL_LABEL_COLUMNS.values())
    published = published.set_index("StudyInstanceUID")[label_cols]
    published.columns = list(OFFICIAL_LABEL_COLUMNS.keys())
    return published


def load_gold_labels(raw_dir):
    train = pd.read_csv(raw_dir / "train.csv")
    label_cols = list(OFFICIAL_LABEL_COLUMNS.values())
    gold_mask = train[label_cols].notna().all(axis=1)
    gold = train.loc[gold_mask, ["StudyInstanceUID"] + label_cols].set_index("StudyInstanceUID")
    gold.columns = list(OFFICIAL_LABEL_COLUMNS.keys())
    return gold


train_csv = pd.read_csv(RAW_DIR / "train.csv")
reports = train_csv.set_index("StudyInstanceUID")[["Report"]]
gold = load_gold_labels(RAW_DIR)
published = load_published_labels(PUBLISHED_LABELS_PATH)

missing = set(train_csv["StudyInstanceUID"]) - set(published.index)
print("train.csv studies missing from published labels:", len(missing))
assert len(missing) == 0

is_gold = reports.index.isin(gold.index)
label_table = published.reindex(reports.index)[FINDINGS].copy()
label_table.loc[gold.index, FINDINGS] = gold[FINDINGS]
label_table["is_gold"] = is_gold
print(label_table.shape, "gold rows:", label_table["is_gold"].sum())

## Folds: report-template + scanner-fingerprint grouping (A0)

Real DICOM header scan across all 4,407 studies - header-only
(`stop_before_pixels=True`), cheap even at this count. Saves
`fold_assignments.csv` as a Kaggle output so a future notebook doesn't
need to redo this scan.

In [ ]:
def report_group_key(report_text):
    if not isinstance(report_text, str):
        normalized = ""
    else:
        t = unicodedata.normalize("NFKD", report_text.lower())
        t = "".join(ch for ch in t if not unicodedata.combining(ch))
        normalized = re.sub(r"\s+", " ", t).strip()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


SCANNER_FINGERPRINT_TAGS = (
    "Manufacturer", "ManufacturerModelName", "InstitutionName",
    "DeviceSerialNumber", "MagneticFieldStrength", "StationName",
)


def build_scanner_fingerprints(raw_dir, split="train"):
    series = pd.read_csv(raw_dir / f"{split}_series.csv")
    first_series = series.drop_duplicates("StudyInstanceUID", keep="first")
    fingerprints = {}
    for row in first_series.itertuples(index=False):
        series_dir = raw_dir / f"{split}_series" / row.StudyInstanceUID / row.SeriesInstanceUID
        files = sorted(series_dir.glob("*.dcm"))
        if not files:
            fingerprints[row.StudyInstanceUID] = None
            continue
        ds = pydicom.dcmread(files[0], stop_before_pixels=True)
        fingerprints[row.StudyInstanceUID] = tuple(
            str(getattr(ds, tag, None)) for tag in SCANNER_FINGERPRINT_TAGS
        )
    result = pd.Series(fingerprints, name="scanner_fingerprint")
    result.index.name = "StudyInstanceUID"
    return result


def build_group_ids(*group_key_series):
    index = group_key_series[0].index
    parent = {i: i for i in index}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for keys in group_key_series:
        valid = keys.dropna()
        for _, idx in valid.groupby(valid).groups.items():
            idx = list(idx)
            for other in idx[1:]:
                union(idx[0], other)

    return pd.Series({i: find(i) for i in index}, name="group_id")


t0 = time.time()
scanner_fp = build_scanner_fingerprints(RAW_DIR, split="train")
print(f"scanner fingerprints: {time.time() - t0:.1f}s for {len(scanner_fp)} studies")

group_keys = reports["Report"].apply(report_group_key)
group_ids = build_group_ids(group_keys, scanner_fp.reindex(reports.index))

gkf = GroupKFold(n_splits=CV_FOLDS)
fold = pd.Series(-1, index=reports.index, dtype=int)
for fold_idx, (_, val_idx) in enumerate(gkf.split(reports, groups=group_ids.to_numpy())):
    fold.iloc[val_idx] = fold_idx
label_table["fold"] = fold
print(label_table["fold"].value_counts().sort_index())

label_table.to_csv("/kaggle/working/fold_assignments.csv")
print("saved fold_assignments.csv")

## Cache dataset

Opens all 4 train shards as memmaps (never materialises the ~12GB cache
in RAM). `group_index=1` selects the centre anchor's 3 slices, mirroring
`src/features.py::select_group`'s indexing exactly.

In [ ]:
class SlotCacheDataset(torch.utils.data.Dataset):
    def __init__(self, cache_dir, shards, labels_df, group_index=GROUP_INDEX, study_ids=None):
        '''study_ids: optional subset to restrict this dataset to (e.g. the
        8 studies of a pre-flight smoke test) - shards are always opened in
        full (cheap, memmap only), then filtered down to this subset before
        the labels_df coverage check below, so labels_df only needs to
        cover the subset, not every study in the loaded shards.'''
        self.group_index = group_index
        caches, masks, all_study_ids, shard_of, local_idx = [], [], [], [], []
        for shard in shards:
            cache = np.load(cache_dir / f"{shard}_cache.npy", mmap_mode="r")
            mask = np.load(cache_dir / f"{shard}_mask.npy")
            studies = pd.read_csv(cache_dir / f"{shard}_studies.csv")
            caches.append(cache)
            masks.append(mask)
            all_study_ids.append(studies["StudyInstanceUID"].to_numpy())
            shard_of.append(np.full(len(studies), len(caches) - 1))
            local_idx.append(np.arange(len(studies)))

        self.caches = caches
        mask_all = np.concatenate(masks, axis=0).astype(np.float32)
        study_ids_all = np.concatenate(all_study_ids)
        shard_of_all = np.concatenate(shard_of)
        local_idx_all = np.concatenate(local_idx)

        if study_ids is not None:
            keep = np.isin(study_ids_all, np.asarray(list(study_ids)))
            mask_all, study_ids_all = mask_all[keep], study_ids_all[keep]
            shard_of_all, local_idx_all = shard_of_all[keep], local_idx_all[keep]

        self.mask = mask_all
        self.study_ids = study_ids_all
        self.shard_of = shard_of_all
        self.local_idx = local_idx_all

        aligned = labels_df.reindex(self.study_ids)[FINDINGS]
        if aligned.isna().any().any():
            missing = self.study_ids[aligned.isna().any(axis=1).to_numpy()]
            raise ValueError(f"{len(missing)} cache studies missing labels, e.g. {missing[:5]}")
        self.labels = aligned.to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, i):
        shard_idx, row = self.shard_of[i], self.local_idx[i]
        full = self.caches[shard_idx][row]  # (6, 9, 224, 224) uint8
        g = self.group_index
        selected = full[:, g * SLOT_CACHE_GROUP_SIZE:(g + 1) * SLOT_CACHE_GROUP_SIZE]
        images = torch.from_numpy(np.ascontiguousarray(selected)).float() / 255.0
        mask = torch.from_numpy(self.mask[i])
        label = torch.from_numpy(self.labels[i])
        return images, mask, label


sanity_ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS[:1], label_table)
images, mask, label = sanity_ds[0]
print("images:", images.shape, images.dtype, "mask:", mask.shape, "label:", label.shape)
assert images.shape == (6, 3, 224, 224)
assert not torch.isnan(images).any()
print("SlotCacheDataset sanity check OK")

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "timm"], check=True)
import timm
print("timm:", timm.__version__)

## Model: DINOv2-small backbone + masked_finding_attention

`masked_finding_attention` below is a hand-kept copy of
`src/model.py::masked_finding_attention` (already unit-tested on
synthetic embeddings, tests/test_model.py) - same function, duplicated
here since this notebook can't `import src`.

In [ ]:
def masked_finding_attention(embeddings, mask, query, head_weight, head_bias):
    if not (mask.sum(dim=1) > 0).all():
        raise ValueError("masked_finding_attention: a row has 0 present slots")
    scores = torch.einsum("od,bsd->bos", query, embeddings) / (embeddings.shape[-1] ** 0.5)
    expanded_mask = mask.unsqueeze(1).expand(-1, query.shape[0], -1)
    scores = scores.masked_fill(expanded_mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    context = torch.einsum("bos,bsd->bod", weights, embeddings)
    logits = (context * head_weight.unsqueeze(0)).sum(-1) + head_bias
    if torch.isnan(logits).any() or torch.isinf(logits).any():
        raise RuntimeError("masked_finding_attention produced NaN/Inf logits")
    return logits


class SlotAttentionModel(nn.Module):
    def __init__(self, n_findings=len(FINDINGS), n_slots=len(SLOT_NAMES),
                 backbone_name="vit_small_patch14_dinov2.lvd142m", unfreeze_last=6):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, num_classes=0, img_size=224,
        )
        embed_dim = self.backbone.num_features
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.blocks[-unfreeze_last:]:
            for p in block.parameters():
                p.requires_grad = True

        self.query = nn.Parameter(torch.randn(n_findings, embed_dim) * (embed_dim ** -0.5))
        self.heads = nn.Linear(embed_dim, n_findings)
        self.embed_dim = embed_dim
        self.n_findings = n_findings
        self.n_slots = n_slots

    def forward(self, slot_images, slot_mask):
        B, S, C, H, W = slot_images.shape
        if (S, C, H, W) != (self.n_slots, 3, 224, 224):
            raise ValueError(
                f"expected slot_images (*, {self.n_slots}, 3, 224, 224), got {tuple(slot_images.shape)}"
            )
        if tuple(slot_mask.shape) != (B, S):
            raise ValueError(f"expected slot_mask ({B}, {S}), got {tuple(slot_mask.shape)}")

        flat = slot_images.view(B * S, C, H, W)
        embeddings = self.backbone(flat).view(B, S, self.embed_dim)
        return masked_finding_attention(
            embeddings, slot_mask, self.query, self.heads.weight, self.heads.bias
        )


print("SlotAttentionModel defined - instantiating to confirm it loads real DINOv2 weights...")
_smoke_model = SlotAttentionModel()
n_trainable = sum(p.numel() for p in _smoke_model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in _smoke_model.parameters())
print(f"embed_dim={_smoke_model.embed_dim}, trainable params={n_trainable:,} / {n_total:,}")
del _smoke_model

## Evaluation helpers

Hand-kept copies of `src/evaluate.py::macro_roc_auc`/`per_finding_roc_auc`
(unchanged), plus a confident-subset filter for the weak gauge (spec
section 5) - published labels are continuous, `roc_auc_score` needs
binary ground truth, so filter to confident rows (`<=0.15` or `>=0.85`)
before scoring that gauge.

In [ ]:
def per_finding_roc_auc(y_true, y_pred):
    # y_true.nunique() < 2 -> ROC AUC is mathematically undefined for that
    # column (e.g. a rare finding with 0 positives among a fold's ~17 gold
    # studies, a real possibility at this sample size, not a bug) - checked
    # explicitly rather than relying on roc_auc_score's own behaviour,
    # which varies by sklearn version (raise vs. warn-and-return-nan).
    scores = {}
    for c in y_true.columns:
        if y_true[c].nunique() < 2:
            scores[c] = float("nan")
        else:
            scores[c] = roc_auc_score(y_true[c], y_pred[c])
    return pd.Series(scores)


def macro_roc_auc(y_true, y_pred):
    per_finding = per_finding_roc_auc(y_true, y_pred)
    undefined = per_finding[per_finding.isna()]
    if len(undefined) > 0:
        print(f"  (macro_roc_auc: {len(undefined)} finding(s) undefined this fold "
              f"- {list(undefined.index)}, excluded from the mean, not treated as 0)")
    return float(per_finding.mean())  # pd.Series.mean() skips NaN by default

## Pre-flight: overfit 8 real studies

Standard wiring sanity check before spending real training time - if the
model can't drive the loss near zero on 8 memorised examples, something
upstream (label alignment, a frozen parameter that should be trainable,
a sign error) is broken, and it's far cheaper to find that here than
after a multi-hour run.

In [ ]:
model = SlotAttentionModel().to(DEVICE)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                         lr=1e-3, weight_decay=0.02)

tiny_studies = label_table.index[:8]
tiny_ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS, label_table, study_ids=tiny_studies)
tiny_loader = torch.utils.data.DataLoader(tiny_ds, batch_size=8, shuffle=False)
images, mask, labels = next(iter(tiny_loader))
images, mask, labels = images.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)

# BCE against continuous (not 0/1) targets has a non-zero floor: the
# per-element binary entropy H(y) of the target itself - even a
# prediction that matches y exactly still incurs
# -[y*log(y) + (1-y)*log(1-y)]. Measured 2026-08-26 on real 8-study
# batches (two separate random inits): loss reliably converges to the
# floor within ~500 steps (one run needed the full 500, oscillating
# through steps 200-300 before settling; a shorter 300-step budget was
# tried first and was NOT reliably enough - real run-to-run variance,
# not a fixed number to shave down further) - so "overfit" here means
# "reach the floor given enough steps", not "reach near-zero".
eps = 1e-7
loss_floor = -(labels * torch.log(labels.clamp(eps, 1)) +
               (1 - labels) * torch.log((1 - labels).clamp(eps, 1))).mean().item()
print(f"loss floor for this batch (soft-label entropy): {loss_floor:.4f}")

print("pre-flight: overfitting 8 real studies...")
model.train()
for step in range(500):
    opt.zero_grad()
    loss = F.binary_cross_entropy_with_logits(model(images, mask), labels)
    loss.backward()
    opt.step()
    if step % 100 == 0:
        print(f"  step {step}: loss={loss.item():.4f}")
print(f"final pre-flight loss: {loss.item():.4f} (floor: {loss_floor:.4f})")
assert loss.item() < loss_floor + 0.02, (
    f"model failed to reach the soft-label loss floor ({loss_floor:.4f}) on 8 real "
    f"studies - stop and debug before the real run"
)
print("pre-flight OK")
del model, opt

## Full single-fold (fold 0) training run

Hyperparameters sourced from `data/raw/_reference_kernels/rsna-knee-500gb-to-11gib-cpu-pixel-cache.ipynb`
(same competition, same backbone family) - see RESOURCES.md.

In [ ]:
FOLD = 0
train_idx = np.flatnonzero(label_table["fold"].to_numpy() != FOLD)
val_idx = np.flatnonzero(label_table["fold"].to_numpy() == FOLD)
val_labels = label_table.iloc[val_idx].reset_index()
val_is_gold = val_labels["is_gold"].to_numpy()
print(f"fold {FOLD}: {len(train_idx)} train / {len(val_idx)} val ({val_is_gold.sum()} gold in val)")

full_ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS, label_table)
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(full_ds, train_idx.tolist()), batch_size=8, shuffle=True, num_workers=2,
)
val_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(full_ds, val_idx.tolist()), batch_size=8, shuffle=False, num_workers=2,
)

model = SlotAttentionModel().to(DEVICE)
backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone")]
head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone")]
opt = torch.optim.AdamW([
    {"params": backbone_params, "lr": 8e-6},
    {"params": head_params, "lr": 1e-3},
], weight_decay=0.02)

EPOCHS = 12
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=[8e-6, 1e-3], total_steps=EPOCHS * len(train_loader)
)

best_gold_auc = -1.0
for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    for images, mask, labels in train_loader:
        images, mask, labels = images.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(images, mask), labels)
        loss.backward()
        opt.step()
        scheduler.step()

    model.eval()
    probs = []
    with torch.no_grad():
        for images, mask, _ in val_loader:
            images, mask = images.to(DEVICE), mask.to(DEVICE)
            probs.append(torch.sigmoid(model(images, mask)).cpu().numpy())
    val_pred = pd.DataFrame(np.concatenate(probs), columns=FINDINGS)
    gold_auc = macro_roc_auc(val_labels.loc[val_is_gold, FINDINGS], val_pred[val_is_gold])
    print(f"epoch {epoch}: {time.time() - t0:.0f}s, val gold macro-AUC={gold_auc:.4f}")

    if gold_auc > best_gold_auc:
        best_gold_auc = gold_auc
        torch.save(model.state_dict(), "/kaggle/working/a2_v1_fold0_best.pt")
        print("  new best, checkpoint saved")

print(f"\nbest gold macro-AUC (fold {FOLD}): {best_gold_auc:.4f}")

## Final report: reload the best checkpoint, score both gauges

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/a2_v1_fold0_best.pt"))
model.eval()
probs = []
with torch.no_grad():
    for images, mask, _ in val_loader:
        images, mask = images.to(DEVICE), mask.to(DEVICE)
        probs.append(torch.sigmoid(model(images, mask)).cpu().numpy())
val_pred = pd.DataFrame(np.concatenate(probs), columns=FINDINGS)

gold_per_finding = per_finding_roc_auc(val_labels.loc[val_is_gold, FINDINGS], val_pred[val_is_gold])
print("gold per-finding AUC:\n", gold_per_finding)
print("gold macro AUC:", float(gold_per_finding.mean()))

weak_mask = ~val_is_gold
weak_true = val_labels.loc[weak_mask, FINDINGS].reset_index(drop=True)
weak_pred = val_pred[weak_mask].reset_index(drop=True)
confident = (weak_true <= 0.15) | (weak_true >= 0.85)
keep = confident.all(axis=1)
n_confident = int(keep.sum())
print(f"\nweak gauge: {n_confident} / {len(weak_true)} weak val studies confident enough")
if n_confident > 0:
    weak_true_bin = (weak_true[keep] >= 0.5).astype(float).reset_index(drop=True)
    weak_pred_conf = weak_pred[keep].reset_index(drop=True)
    print("weak gauge macro AUC (confident subset):", macro_roc_auc(weak_true_bin, weak_pred_conf))

print("\nhistorical real-leaderboard reference (Fase 5, informal sanity check only): 0.596")

## Real output (fill in after running on Kaggle)

_Placeholder - paste the real printed output here once this notebook
has actually been run on Kaggle with the cache Dataset attached and GPU
enabled, same convention as `00v2`/`03v2`/`04v2`. This is where
`per_label_gate()` (already in `src/evaluate.py`, no change needed)
gets applied to *future* A2 candidates against this run's number - this
run has nothing to gate against yet, it establishes the baseline._